In [ ]:
import os
import sys, pathlib
import numpy as np

sys.path.append(os.path.join(os.path.expanduser('~'), 'Programming', 'In_Vivo','physion', 'src'))

#import physion
#import physion.utils
#import physion.analysis
#import physion.dataviz

from physion.utils import plot_tools as pt
from physion.analysis.read_NWB import Data, scan_folder_for_NWBfiles
from physion.dataviz.imaging import show_CaImaging_FOV
from physion.dataviz.raw import plot as plot_raw, find_default_plot_settings


from physion.dataviz import tools as dv_tools

In [ ]:
names = ["2024_08_27-12-21-14.nwb", 
         "2024_08_27-12-46-41.nwb",
         "2024_09_12-14-57-34.nwb", 
         "2024_09_12-15-24-47.nwb", 
         "2024_09_12-15-50-12.nwb", 
         "2024_10_07-15-03-40.nwb", 
         "2024_10_07-16-26-15.nwb", 
         "2024_10_07-17-18-53.nwb", 
         "2024_10_11-14-13-22.nwb", 
         "2024_10_11-14-57-27.nwb", 
         "2024_10_11-15-46-32.nwb", 
         "2024_10_11-16-44-26.nwb", 
         "2024_10_11-17-26-55.nwb",
         "2024_10_11-18-24-27.nwb"]

filenames = []

for i in range(len(names)):
    filename = os.path.join(os.path.expanduser('~'), 'Desktop', 'NWBs', names[i])
    filenames.append(filename)


In [ ]:
DATA = []
for filename in filenames:
    print(filename)
    data = Data(filename, verbose=False)
    DATA.append(data)

## Plot MeanImg, MaxProj, ROIs

In [ ]:
for data in DATA:
    #data.build_rawFluo(verbose=False)
    data.build_dFoF(verbose=False)
    
    fig, AX = pt.figure(axes=(3,1), figsize=(1.4,3), wspace=0.15)

    show_CaImaging_FOV(data, key='meanImg', 
                       cmap=pt.get_linear_colormap('k', 'tab:green'),
                       NL=2, # non-linearity to normalize image
                       ax=AX[0])
    show_CaImaging_FOV(data, key='max_proj', 
                       cmap=pt.get_linear_colormap('k', 'tab:green'),
                       NL=2, # non-linearity to normalize image
                       ax=AX[1])
    show_CaImaging_FOV(data, key='meanImg', 
                       cmap=pt.get_linear_colormap('k', 'tab:green'),
                       NL=2,
                       roiIndices=range(data.nROIs), 
                       ax=AX[2])
    # save on desktop
    fig.savefig(os.path.join(os.path.expanduser('~'), 'Desktop', 'FOVs', f'FOV_{os.path.basename(filename)}.png'))

In [ ]:
settings = {'Locomotion': {'fig_fraction': 1,
                           'subsampling': 1,
                           'color': '#1f77b4'},
            'FaceMotion': {'fig_fraction': 1,
                           'subsampling': 1,
                           'color': 'purple'},
            'Pupil': {'fig_fraction': 2,
                      'subsampling': 1,
                      'color': '#d62728'},
            'CaImaging': {'fig_fraction': 10,
                          'subsampling': 1,
                          'subquantity': 'dF/F',
                          'roiIndices': np.random.choice(np.arange(data.nROIs), np.min([20,data.nROIs]), replace=False),
                          'color': '#2ca02c'}
           }
for data in DATA:
    try:
        fig2, AX = pt.figure(axes=(1,1), figsize=(3,5), wspace=0.15)
        plot_raw(data, tlim=[0, data.t_dFoF[-1]], settings=settings, ax=AX)
        fig2.savefig(os.path.join(os.path.expanduser('~'), 'Desktop', 'Traces', f'Trace_{data.filename}.png'))
    except Exception as e: 
        print(data.filename)
        print(f"Problem : {e}")


In [ ]:
settings = {'CaImagingSum': {'fig_fraction': 10,
                             'subsampling': 1,
                             'subquantity': 'dF/F'}
           }
for data in DATA:
    try:
        fig2, AX = pt.figure(axes=(1,1), figsize=(3,5), wspace=0.15)
        plot_raw(data, tlim=[0, data.t_dFoF[-1]], settings=settings, ax=AX)
        fig2.savefig(os.path.join(os.path.expanduser('~'), 'Desktop', 'Traces', f'Trace_{data.filename}.png'))
    except Exception as e: 
        print(data.filename)
        print(f"Problem : {e}")


In [ ]:
data.t_dFoF

In [ ]:
for filename in filenames: 
    try:
        data = physion.analysis.read_NWB.Data(filename,
                                             verbose=False)
        episodes = physion.analysis.process_NWB.EpisodeData(data, 
                                                            quantities=['dFoF'],
                                                            protocol_id=0)
        plot_props = dict(#column_key='contrast',
                      Xbar=1, Xbar_label='1s',
                      with_annotation=True,
                      figsize=(9,1.8))
        
        fig, AX = physion.dataviz.episodes.trial_average.plot(episodes,
                                                          roiIndices='all',
                                                          **plot_props)
    except Exception as e: 
        print(filename)
        print(f"Problem : {e}")

In [ ]:
#to add to python file dataviz/imaging


'''
def add_CaImaging_mean(data, tlim, ax,
                     fig_fraction_start=0., fig_fraction=1., color='green',
                     subquantity='Fluorescence', subsampling=1,
                     name='Mean [Ca]'):
    
    if (subquantity in ['dF/F', 'dFoF']) and (not hasattr(data, 'dFoF')):
        data.build_dFoF()
        
    i1, i2 = dv_tools.convert_times_to_indices(*tlim, data.Neuropil, axis=1)
    t = np.array(data.Neuropil.timestamps[:])[np.arange(i1,i2)][::subsampling]
    
    if (subquantity in ['dF/F', 'dFoF']):
        y = data.dFoF.mean(axis=0)[np.arange(i1,i2)][::subsampling]
    else:
        y = data.rawFluo.mean(axis=0)[np.arange(i1,i2)][::subsampling]

    
'''
#    dv_tools.plot_scaled_signal(data, ax, t, y, tlim, 1., fig_fraction, fig_fraction_start, color=color,
#                            scale_unit_string=('%.0fdF/F' if subquantity in ['dF/F', 'dFoF'] else ''))
'''#not adding something for the scale_side was creating bugs (before fig_fraction) SOFIA
    
    dv_tools.plot_scaled_signal(data, ax, t, y, tlim, 1., 'left', fig_fraction, fig_fraction_start, color=color,
                            scale_unit_string=('%.0fdF/F' if subquantity in ['dF/F', 'dFoF'] else ''))

    dv_tools.add_name_annotation(data, ax, name, tlim, fig_fraction, fig_fraction_start, color=color)
'''
'''
def plot_test(data,
         tlim=[0,100],
         settings = {},
         figsize=(9,6), 
         Tbar=0., zoom_area=None,
         ax=None):

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = None

    fig_fraction_full, fstart = np.sum([settings[key]['fig_fraction'] for key in settings]), 0

    for key in settings:
        settings[key]['fig_fraction_start'] = fstart
        settings[key]['fig_fraction'] = settings[key]['fig_fraction']/fig_fraction_full
        fstart += settings[key]['fig_fraction']

    add_CaImaging_mean(data, tlim, ax, **settings['CaImaging_mean'])

    '''
    #not managed to make it work
    for key in settings:
        print(key)
        print('add_%s' %key)
        exec('add_%s(data, tlim, ax, **settings[key])' % key)
    '''

    # time scale bar
    if Tbar==0.:
        Tbar = np.max([int((tlim[1]-tlim[0])/30.), 1])

    ax.plot([dv_tools.shifted_start(tlim), dv_tools.shifted_start(tlim)+Tbar], [1.,1.], lw=1, color='k')
    ax.annotate((' %is' % Tbar if Tbar>=1 else  '%.1fs' % Tbar) ,
                [dv_tools.shifted_start(tlim), 1.02], color='k')#, fontsize=9)

    ax.axis('off')
    ax.set_xlim([dv_tools.shifted_start(tlim)-0.01*(tlim[1]-tlim[0]),tlim[1]+0.01*(tlim[1]-tlim[0])])
    ax.set_ylim([-0.05,1.05])

    if zoom_area is not None:
        ax.fill_between(zoom_area, [0,0], [1,1],  color='k', alpha=.2, lw=0)

    return fig, ax
'''